In [7]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import re
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, precision_recall_curve,
    classification_report
)

# 可选：LightGBM 更强
import lightgbm as lgb

RANDOM_STATE = 42

In [8]:

# =========================
# 1) 读取主表与距离映射
# =========================
df = pd.read_csv("panel_data_1111/after_merge_1111.csv")
dist_bj_sh = pd.read_csv("dist_origin_to_bj_sh.csv")  # 只含 hs_residence, dist_to_bj, dist_to_sh

# 仅保留跨省流动样本
df = df[df["Migrate"] == 1].copy()
df = df[df["hs_residence"] != df["pro_code"]].copy()
if "Migrate_1" in df.columns:
    df = df[df["Migrate_1"] == 1].copy()

# 标签：是否去北京/上海
df["y_bjsh"] = df["pro_code"].isin([11, 31]).astype(int)

In [9]:

# =========================
# 2) 去除泄漏变量
# =========================
drop_cols_leak = [
    "pro_name","city_clean","is_beijing","is_shanghai",
    "pro_name_true","English_name","gdp_after_move","migration_distance_km"
]
drop_cols_leak += [c for c in df.columns if c.endswith("_a")]
df = df.drop(columns=[c for c in drop_cols_leak if c in df.columns])

In [10]:

# =========================
# 3) 合并“原籍→北京/上海”距离（常量，不泄漏）
# =========================
df["hs_residence"] = pd.to_numeric(df["hs_residence"], errors="coerce").astype("Int64")
dist_bj_sh["hs_residence"] = pd.to_numeric(dist_bj_sh["hs_residence"], errors="coerce").astype("Int64")
df = df.merge(dist_bj_sh, on="hs_residence", how="left")

# 衍生距离特征（不泄漏）
df["dist_bj_minus_sh"] = df["dist_to_bj"] - df["dist_to_sh"]
df["dist_min_bj_sh"]   = df[["dist_to_bj","dist_to_sh"]].min(axis=1)
df["dist_ratio_bj_over_sh"] = (df["dist_to_bj"] + 1) / (df["dist_to_sh"] + 1)

In [11]:

# =========================
# 4) 静态宏观差值：北京/上海静态值 - 原籍_b
# =========================
# external_data2.xlsx：你的静态宏观来源（和之前 on="hs_residence" 的那张）
# 若文件不存在或列名不一致，可跳过本段（保持鲁棒）
def safe_name(name: str) -> str:
    return re.sub(r"[^0-9a-zA-Z_]+", "_", str(name)).strip("_").lower()

macro_b_cols = [c for c in df.columns if c.endswith("_b")]
try:
    ext2 = pd.read_excel("external_data2.xlsx")
    # 兼容 ext2 使用 hs_residence 或 pro_code 作为省份列
    key_col = "hs_residence" if "hs_residence" in ext2.columns else ("pro_code" if "pro_code" in ext2.columns else None)
    if key_col is not None:
        row_bj = ext2.loc[ext2[key_col] == 11]
        row_sh = ext2.loc[ext2[key_col] == 31]
        if not row_bj.empty and not row_sh.empty:
            row_bj = row_bj.iloc[0]
            row_sh = row_sh.iloc[0]
            for col in macro_b_cols:
                base = col[:-2]  # 去掉 _b
                if base in ext2.columns:
                    v_bj = row_bj[base]
                    v_sh = row_sh[base]
                    df[f"delta_{safe_name(base)}_bj"] = v_bj - df[col]
                    df[f"delta_{safe_name(base)}_sh"] = v_sh - df[col]
        else:
            print("[warn] external_data2.xlsx 中未找到北京/上海两行，跳过静态差值构造。")
    else:
        print("[warn] external_data2.xlsx 中没有 hs_residence/pro_code 键，跳过静态差值构造。")
except FileNotFoundError:
    print("[warn] 未找到 external_data2.xlsx，跳过静态差值构造。")

In [12]:

# =========================
# 5) 年度 GDP 差值（可选，若有年度面板；无则自动跳过）
# =========================
# 用年度面板获得北京/上海当年 GDP，再与 gdp_before_move 做 log 差（非必须）
try:
    gdp_panel = pd.read_excel("china_correct_panel.xlsx")  # 需要包含列：pro_code, year, real_GDP
    gdp_panel = gdp_panel.rename(columns={"year":"migration_year"})
    bj = gdp_panel[gdp_panel["pro_code"] == 11][["migration_year","real_GDP"]].rename(columns={"real_GDP":"gdp_bj"})
    sh = gdp_panel[gdp_panel["pro_code"] == 31][["migration_year","real_GDP"]].rename(columns={"real_GDP":"gdp_sh"})
    df = df.merge(bj, on="migration_year", how="left")
    df = df.merge(sh, on="migration_year", how="left")

    if "gdp_before_move" in df.columns:
        df["log_gdp_before"] = np.log(df["gdp_before_move"].replace({0: np.nan}))
        df["log_gdp_bj"] = np.log(df["gdp_bj"].replace({0: np.nan}))
        df["log_gdp_sh"] = np.log(df["gdp_sh"].replace({0: np.nan}))
        df["delta_log_gdp_bj_vs_origin"] = df["log_gdp_bj"] - df["log_gdp_before"]
        df["delta_log_gdp_sh_vs_origin"] = df["log_gdp_sh"] - df["log_gdp_before"]
    else:
        print("[info] 未找到 gdp_before_move，跳过年度 GDP 差值。")
except FileNotFoundError:
    print("[info] 未找到 china_correct_panel.xlsx，跳过年度 GDP 差值。")


In [13]:

# =========================
# 6) 个人特征与金额变换
# =========================
# 金额类 log1p
for c in ["income_total_m_win","exp_total_m_win","food_exp_m_win","income_to_home_win","rent_m_win"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
        df[f"log1p_{c}"] = np.log1p(df[c])

# 构建特征清单
label = "y_bjsh"
cat_cols = []
if "employment_group" in df.columns:
    cat_cols.append("employment_group")

# 不将 hs_residence 用作特征（仅用于合并映射）
exclude_cols = {label, "hs_residence", "gdp_bj", "gdp_sh", "log_gdp_before", "log_gdp_bj", "log_gdp_sh"}
feature_cols = [c for c in df.columns if c not in exclude_cols]

# 可选：去掉高缺失（>80%）的列，减少噪声
na_rate = df[feature_cols].isna().mean()
drop_high_na = na_rate[na_rate > 0.8].index.tolist()
if drop_high_na:
    print(f"[info] 丢弃高缺失特征（>{0.8:.0%}）：{len(drop_high_na)} 列")
    feature_cols = [c for c in feature_cols if c not in drop_high_na]

In [14]:

# =========================
# 7) 随机划分（打乱 + Stratified）
# =========================
X = df[feature_cols].copy()
y = df[label].astype(int).values

# 70% 训练，15% 验证，15% 测试
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_va, X_te, y_va, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_tmp
)

In [15]:

# =========================
# 8) 预处理与基线模型：L1 Logistic
# =========================
from pandas.api.types import is_numeric_dtype

# 先得到 feature_cols（保持你原有逻辑）
# feature_cols = [...]

# 1) 可选：自动把“看起来像数值”的 object 列转为数值
obj_cols = df[feature_cols].select_dtypes(include=["object"]).columns.tolist()
def coerce_numeric_like(df, cols, min_numeric_rate=0.9):
    for c in cols:
        s = df[c].astype(str).str.replace(",", "").str.strip()
        s_num = pd.to_numeric(s, errors="coerce")
        # 若 >=90% 的值能成功转成数值，则视为数值列
        if s_num.notna().mean() >= min_numeric_rate:
            df[c] = s_num
coerce_numeric_like(df, obj_cols)

# 2) 依据 dtype 动态划分 数值/类别 列（避免把字符串放进“数值通道”）
num_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in feature_cols if c not in num_cols]

print(f"[info] 数值特征数: {len(num_cols)}, 类别特征数: {len(cat_cols)}")
# 若某些你明确知道是类别列（如 employment_group）不在 cat_cols，可以手工补充：
for must_cat in ["employment_group"]:
    if must_cat in feature_cols and must_cat not in cat_cols:
        if must_cat in num_cols:
            num_cols.remove(must_cat)
        cat_cols.append(must_cat)

# 3) 构建 ColumnTransformer（如果某一类为空也能正常工作）
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

transformers = []
if len(num_cols) > 0:
    transformers.append(("num", SimpleImputer(strategy="median"), num_cols))
if len(cat_cols) > 0:
    transformers.append((
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore"))
        ]),
        cat_cols
    ))

preprocess = ColumnTransformer(transformers=transformers, remainder="drop")

[info] 数值特征数: 93, 类别特征数: 2


In [18]:
X_te.columns

Index(['pro_code', 'male', 'birth_year', 'age', 'is_han', 'high_school',
       'junior_college', 'bachelor', 'graduate', 'rural', 'Migrate',
       'Migrate_1', 'Migrate_2', 'Migrate_3', 'migration_year',
       'migration_interval', 'marriage', 'employed', 'income_total_m',
       'exp_total_m', 'food_exp_m', 'income_to_home', 'rent_m',
       'income_total_m_win', 'exp_total_m_win', 'food_exp_m_win',
       'income_to_home_win', 'rent_m_win', 'employment_category',
       'employment_group', 'workdays_w', 'workhours_d', 'hours_per_week',
       'hours_per_week_filled', 'marriage_year', 'length_marriage',
       'kids_number', 'birth_here', 'Pension_Insurance', 'Medical_Insurance',
       'Work_Insurance', 'Unemploy_Insurance', 'Maternity_Insurance',
       'Housing_Fund', 'Happiness', 'year', 'lowest_temp(Jan)_b',
       'average_temp_b', 'highest_temp(July)_b', 'precipitation(mm)_b',
       'gdp per capita(k)_b', 'unemployment(%)_b', 'education_budget(10k)_b',
       'marriage(10k)

In [31]:

# ========== 7) 构造标签 ==========
# 3类：0=其他, 1=北京(11), 2=上海(31)
y3 = np.where(df["pro_code"].astype(int)==11, 1,
     np.where(df["pro_code"].astype(int)==31, 2, 0)).astype(int)

# 31类：pro_code → 0..C-1
codes_all = sorted(df["pro_code"].dropna().astype(int).unique().tolist())
code2idx = {c:i for i,c in enumerate(codes_all)}
idx2code = {i:c for c,i in code2idx.items()}
y31 = df["pro_code"].astype(int).map(code2idx).values

# ========== 8) 选择少而精的特征（只取存在的列） ==========
core_feats = [
    # 个人/教育/就业
    "male","age","is_han","rural",
    "high_school","junior_college","bachelor","graduate",
    "employed","employment_group","hours_per_week_filled",
    # 金额（取 log1p 版）
    "log1p_income_total_m_win","log1p_exp_total_m_win","log1p_food_exp_m_win","log1p_income_to_home_win","log1p_rent_m_win",
    # 家庭/婚育/幸福
    "marriage","length_marriage","kids_number","Happiness",
    # 迁移时长
    "migration_interval",
    # 原籍静态宏观（部分）
    "gdp per capita(k)_b","unemployment(%)_b","population(10k)_b","manageable_income_per_capita_b",
    "education_budget(10k)_b","Medical technicians per 10k_b","road_length_per_10K (km)_b",
    # 静态差值（若已生成）
    "delta_gdp_per_capita_k__bj","delta_gdp_per_capita_k__sh",
    "delta_unemployment___bj","delta_unemployment___sh",
    "delta_population_10k__bj","delta_population_10k__sh",
    "delta_manageable_income_per_capita_bj","delta_manageable_income_per_capita_sh",
    # 年度 GDP 差值（若已生成）
    "delta_log_gdp_bj_vs_origin","delta_log_gdp_sh_vs_origin",
    # 距离
    "dist_to_bj","dist_to_sh","dist_bj_minus_sh","dist_min_bj_sh","dist_ratio_bj_over_sh",
]

# 上面静态差值列名可能因为 safe_name 处理略有不同，保险起见：补全所有 delta_* 列
delta_cols = [c for c in df.columns if c.startswith("delta_")]
feature_pool = set(core_feats) | set(delta_cols)
# 移除明显非特征键
feature_pool -= {"pro_code","hs_residence","gdp_bj","gdp_sh","log_gdp_before","log_gdp_bj","log_gdp_sh"}
selected_features = [c for c in feature_pool if c in df.columns]

# 指定类别列（CatBoost）
cat_cols = []
for c in selected_features:
    if df[c].dtype == "object" or str(df[c].dtype).startswith(("string","category")):
        cat_cols.append(c)
# 确保 employment_group 走类别通道
if "employment_group" in selected_features and "employment_group" not in cat_cols:
    cat_cols.append("employment_group")
# 统一类别列为字符串
for c in cat_cols:
    df[c] = df[c].astype("string")

# ========== 9) 随机划分（分层，31类为准），并共用索引给两任务 ==========
X_full = df[selected_features].copy()
y_full_31 = y31.copy()
y_full_3  = y3.copy()

X_tr_all, X_te_hold, y_tr_all_31, y_te_hold_31, idx_tr_all, idx_te_hold = train_test_split(
    X_full, y_full_31, np.arange(len(X_full)),
    test_size=0.15, random_state=RANDOM_STATE, stratify=y_full_31
)
# 验证集
X_tr_31, X_va_31, y_tr_31, y_va_31, idx_tr, idx_va = train_test_split(
    X_tr_all, y_tr_all_31, idx_tr_all,
    test_size=0.1765, random_state=RANDOM_STATE, stratify=y_tr_all_31
)

# 同步得到 3 类标签的划分（使用相同索引）
y_tr_all_3 = y_full_3[idx_tr_all]
y_tr_3 = y_full_3[idx_tr]
y_va_3 = y_full_3[idx_va]
y_te_3 = y_full_3[idx_te_hold]

X_te_31 = X_te_hold.copy()
y_te_31 = y_te_hold_31.copy()

In [4]:

# ========== 10) 训练 CatBoost 多分类（长时间） ==========
from catboost import CatBoostClassifier, Pool

# 3类
tr_pool3 = Pool(X_tr_31, label=y_tr_3, cat_features=cat_cols)  # 训练/验证集用相同划分索引
va_pool3 = Pool(X_va_31, label=y_va_3, cat_features=cat_cols)
te_pool3 = Pool(X_te_31, label=y_te_3, cat_features=cat_cols)

cls_3 = np.unique(y_tr_3)
w_3 = compute_class_weight(class_weight="balanced", classes=cls_3, y=y_tr_3)
class_weights_3 = [w_3[list(cls_3).index(i)] for i in range(3)]

cb3 = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="MultiClass",
    iterations=50000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=6.0,
    random_seed=RANDOM_STATE,
    class_weights=class_weights_3,
    od_type="Iter",
    od_wait=1500,
    task_type="GPU",     # 有GPU可改 "GPU"
    verbose=200
)
cb3.fit(tr_pool3, eval_set=va_pool3)

proba_va3 = cb3.predict_proba(va_pool3)
proba_te3 = cb3.predict_proba(te_pool3)
print("3类验证集 Top-k:", topk_accuracy(y_va_3, proba_va3, ks=(1,2,3)))
print("3类测试集 Top-k:", topk_accuracy(y_te_3, proba_te3, ks=(1,2,3)))
print("3类测试集分类报告（Top-1）：")
pred_te3 = proba_te3.argmax(axis=1)
print(classification_report(y_te_3, pred_te3, digits=3, target_names=["other","beijing","shanghai"]))

0:	learn: 1.0856215	test: 1.0869541	best: 1.0869541 (0)	total: 73.8ms	remaining: 1h 1m 28s
200:	learn: 0.7078139	test: 0.8147916	best: 0.8147916 (200)	total: 3.21s	remaining: 13m 16s
400:	learn: 0.6293240	test: 0.7963346	best: 0.7962638 (396)	total: 6.31s	remaining: 13m
600:	learn: 0.5742714	test: 0.7900575	best: 0.7899720 (598)	total: 9.42s	remaining: 12m 54s
800:	learn: 0.5282420	test: 0.7897778	best: 0.7892652 (783)	total: 12.4s	remaining: 12m 43s
1000:	learn: 0.4885814	test: 0.7939051	best: 0.7892652 (783)	total: 15.6s	remaining: 12m 43s
1200:	learn: 0.4524003	test: 0.7989339	best: 0.7892652 (783)	total: 18.7s	remaining: 12m 39s
1400:	learn: 0.4211384	test: 0.8038789	best: 0.7892652 (783)	total: 21.9s	remaining: 12m 39s
1600:	learn: 0.3932255	test: 0.8111807	best: 0.7892652 (783)	total: 25.1s	remaining: 12m 37s
1800:	learn: 0.3674902	test: 0.8198905	best: 0.7892652 (783)	total: 28.2s	remaining: 12m 35s
2000:	learn: 0.3445282	test: 0.8297885	best: 0.7892652 (783)	total: 31.4s	remain

In [35]:
# 导入 Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

# 重新计算数值和类别列（基于 selected_features）
# 因为上面的特征选择可能改变了 num_cols/cat_cols 的内容，需要重新确认
num_cols_final = [c for c in selected_features if is_numeric_dtype(X_full[c]) and c not in cat_cols]
cat_cols_final = [c for c in selected_features if c in cat_cols]

# 重新构建预处理 ColumnTransformer（基于最终特征集）
transformers_final = []
if len(num_cols_final) > 0:
    transformers_final.append(("num", SimpleImputer(strategy="median"), num_cols_final))
if len(cat_cols_final) > 0:
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
        
    transformers_final.append((
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", ohe)
        ]),
        cat_cols_final
    ))

preprocess_final = ColumnTransformer(transformers=transformers_final, remainder="drop")

print(f"\n[Final Features] Num: {len(num_cols_final)}, Cat: {len(cat_cols_final)}")

# ========== 10) 评估函数（Top-K & Macro F1） ==========

def topk_accuracy(y_true_idx, proba, ks=(1,3,5,10)):
    """计算 Top-K 准确率，适用于多分类."""
    order = np.argsort(-proba, axis=1)
    out = {}
    for k in ks:
        topk = order[:, :k]
        out[k] = float((topk == y_true_idx[:, None]).any(axis=1).mean())
    return out

def report_metrics_31class(name, proba, y_true):
    """报告 Top-K 准确率和 Macro F1 Score."""
    # 计算 Top-k 准确率
    res_topk = topk_accuracy(y_true, proba, ks=(1,3,5,10))
    
    # 预测 Top-1 类别
    y_pred = proba.argmax(axis=1)
    
    # 计算 Macro F1 Score (适用于类别不平衡)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    print(f"[{name}]")
    print(f"  Top-1={res_topk[1]:.4f}  Top-3={res_topk[3]:.4f}  Top-5={res_topk[5]:.4f}  Top-10={res_topk[10]:.4f}")
    print(f"  F1-Macro={f1_macro:.4f}")


# ========== 11) 模型 1: Logistic Regression (31分类) ==========
print("\n" + "="*40)
print("=== 11) Logistic Regression (31-class) ===")
print("="*40)

logit_pipe = Pipeline([
    ("prep", preprocess_final),
    ("clf", LogisticRegression(
        multi_class="multinomial",
        solver="lbfgs",
        max_iter=1000,
        class_weight="balanced", # 处理类别不平衡
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

# 训练模型
logit_pipe.fit(X_tr_31, y_tr_31)

# 评估
proba_va_logit = logit_pipe.predict_proba(X_va_31)
proba_te_logit = logit_pipe.predict_proba(X_te_31)

report_metrics_31class("VALID", proba_va_logit, y_va_31)
report_metrics_31class("TEST",  proba_te_logit, y_te_31)




[Final Features] Num: 58, Cat: 1

=== 11) Logistic Regression (31-class) ===


c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[VALID]
  Top-1=0.1007  Top-3=0.2642  Top-5=0.3615  Top-10=0.5692
  F1-Macro=0.0475
[TEST]
  Top-1=0.1020  Top-3=0.2706  Top-5=0.3684  Top-10=0.5715
  F1-Macro=0.0481


In [36]:

# ========== 12) 模型 2: Random Forest (31分类) ==========
print("\n" + "="*40)
print("=== 12) Random Forest (31-class) ===")
print("="*40)

rf_pipe = Pipeline([
    ("prep", preprocess_final),
    ("clf", RandomForestClassifier(
        n_estimators=500, # 适当减少 n_estimators 以加快运行
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=3,
        max_features="sqrt",
        class_weight="balanced", # 处理类别不平衡
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

# 训练模型
rf_pipe.fit(X_tr_31, y_tr_31)

# 评估
proba_va_rf = rf_pipe.predict_proba(X_va_31)
proba_te_rf = rf_pipe.predict_proba(X_te_31)

report_metrics_31class("VALID", proba_va_rf, y_va_31)
report_metrics_31class("TEST",  proba_te_rf, y_te_31)



=== 12) Random Forest (31-class) ===
[VALID]
  Top-1=0.3442  Top-3=0.6072  Top-5=0.7295  Top-10=0.8714
  F1-Macro=0.2550
[TEST]
  Top-1=0.3429  Top-3=0.5985  Top-5=0.7279  Top-10=0.8735
  F1-Macro=0.2531


In [37]:


# ========== 13) 模型 3: LightGBM (31分类) ==========
print("\n" + "="*40)
print("=== 13) LightGBM (31-class) ===")
print("="*40)

# 修复 LightGBMError: 明确指定类别数
C_31 = len(codes_all)

lgbm_pipe = Pipeline([
    ("prep", preprocess_final),
    ("clf", lgb.LGBMClassifier(
        n_estimators=1000,          # 提升树数量
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="multiclass",
        num_class=C_31,             # 显式指定类别数
        class_weight="balanced",    # 处理类别不平衡
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

# 训练模型
# 注意: LightGBM 会自动处理缺失值，但 Pipeline 中的 SimpleImputer 已经处理了
lgbm_pipe.fit(X_tr_31, y_tr_31)

# 评估
proba_va_lgbm = lgbm_pipe.predict_proba(X_va_31)
proba_te_lgbm = lgbm_pipe.predict_proba(X_te_31)

report_metrics_31class("VALID", proba_va_lgbm, y_va_31)
report_metrics_31class("TEST",  proba_te_lgbm, y_te_31)

# 额外报告：测试集上 LightGBM 的 Top-1 混淆矩阵形状
from sklearn.metrics import confusion_matrix
pred_top1_lgbm = proba_te_lgbm.argmax(axis=1)
cm_lgbm = confusion_matrix(y_te_31, pred_top1_lgbm)
print(f"\n[Test] LightGBM Confusion matrix shape: {cm_lgbm.shape} (31x31)")


=== 13) LightGBM (31-class) ===
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004751 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2266
[LightGBM] [Info] Number of data points in the train set: 34783, number of used features: 61
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Start traini

c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[VALID]
  Top-1=0.3876  Top-3=0.6714  Top-5=0.7869  Top-10=0.9044
  F1-Macro=0.2857
[TEST]
  Top-1=0.3916  Top-3=0.6563  Top-5=0.7785  Top-10=0.9031
  F1-Macro=0.2845

[Test] LightGBM Confusion matrix shape: (30, 30) (31x31)


In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import re
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight # 用于计算CatBoost的权重
from pandas.api.types import is_numeric_dtype

# 引入强大的树模型
import lightgbm as lgb
from catboost import CatBoostClassifier

RANDOM_STATE = 42

# =========================
# 辅助函数
# =========================
def safe_name(name: str) -> str:
    """将字符串转换为安全的文件/变量名格式"""
    return re.sub(r"[^0-9a-zA-Z_]+", "_", str(name)).strip("_").lower()

def coerce_numeric_like(df, cols, min_numeric_rate=0.9):
    """尝试将看起来像数值的 object 列转换为数值型"""
    for c in cols:
        s = df[c].astype(str).str.replace(",", "").str.strip()
        s_num = pd.to_numeric(s, errors="coerce")
        if s_num.notna().mean() >= min_numeric_rate:
            df[c] = s_num

def topk_accuracy(y_true_idx, proba, ks=(1,3,5,10)):
    """计算 Top-K 准确率，适用于多分类"""
    order = np.argsort(-proba, axis=1)
    out = {}
    for k in ks:
        topk = order[:, :k]
        out[k] = float((topk == y_true_idx[:, None]).any(axis=1).mean())
    return out

def report_metrics_31class(name, proba, y_true):
    """报告 Top-K 准确率和 Macro F1 Score"""
    res_topk = topk_accuracy(y_true, proba, ks=(1,3,5,10))
    y_pred = proba.argmax(axis=1)
    # 计算 Macro F1 Score (适用于类别不平衡)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    
    print(f"[{name}]")
    print(f"  Top-1={res_topk[1]:.4f}  Top-3={res_topk[3]:.4f}  Top-5={res_topk[5]:.4f}  Top-10={res_topk[10]:.4f}")
    print(f"  F1-Macro={f1_macro:.4f}")

# =========================
# 1) 数据加载与清洗 (与原代码一致)
# =========================
try:
    df = pd.read_csv("panel_data_1111/after_merge_1111.csv")
    dist_bj_sh = pd.read_csv("dist_origin_to_bj_sh.csv")
except FileNotFoundError as e:
    print(f"错误: 缺少关键文件。请确保以下文件存在: {e.filename}")
    raise

# 仅保留跨省流动样本
df = df[df["Migrate"] == 1].copy()
df = df[df["hs_residence"] != df["pro_code"]].copy()
if "Migrate_1" in df.columns:
    df = df[df["Migrate_1"] == 1].copy()

# 标签：是否去北京/上海 (y_bjsh 用于双分类，这里主要用 y31)
df["y_bjsh"] = df["pro_code"].isin([11, 31]).astype(int)

# =========================
# 2) 特征工程 (与原代码一致)
# =========================

# 2.1) 去除泄漏变量
drop_cols_leak = [
    "pro_name","city_clean","is_beijing","is_shanghai",
    "pro_name_true","English_name","gdp_after_move","migration_distance_km"
]
drop_cols_leak += [c for c in df.columns if c.endswith("_a")]
df = df.drop(columns=[c for c in drop_cols_leak if c in df.columns])

# 2.2) 合并距离特征
df["hs_residence"] = pd.to_numeric(df["hs_residence"], errors="coerce").astype("Int64")
dist_bj_sh["hs_residence"] = pd.to_numeric(dist_bj_sh["hs_residence"], errors="coerce").astype("Int64")
df = df.merge(dist_bj_sh, on="hs_residence", how="left")

df["dist_bj_minus_sh"] = df["dist_to_bj"] - df["dist_to_sh"]
df["dist_min_bj_sh"]   = df[["dist_to_bj","dist_to_sh"]].min(axis=1)
df["dist_ratio_bj_over_sh"] = (df["dist_to_bj"] + 1) / (df["dist_to_sh"] + 1)

# 2.3) 静态宏观差值
macro_b_cols = [c for c in df.columns if c.endswith("_b")]
try:
    ext2 = pd.read_excel("external_data2.xlsx")
    key_col = "hs_residence" if "hs_residence" in ext2.columns else ("pro_code" if "pro_code" in ext2.columns else None)
    if key_col is not None:
        row_bj = ext2.loc[ext2[key_col] == 11].iloc[0]
        row_sh = ext2.loc[ext2[key_col] == 31].iloc[0]
        for col in macro_b_cols:
            base = col[:-2]
            if base in ext2.columns:
                df[f"delta_{safe_name(base)}_bj"] = row_bj[base] - df[col]
                df[f"delta_{safe_name(base)}_sh"] = row_sh[base] - df[col]
except Exception:
    pass # 忽略文件或行缺失

# 2.4) 年度 GDP 差值
try:
    gdp_panel = pd.read_excel("china_correct_panel.xlsx").rename(columns={"year":"migration_year"})
    bj = gdp_panel[gdp_panel["pro_code"] == 11][["migration_year","real_GDP"]].rename(columns={"real_GDP":"gdp_bj"})
    sh = gdp_panel[gdp_panel["pro_code"] == 31][["migration_year","real_GDP"]].rename(columns={"real_GDP":"gdp_sh"})
    df = df.merge(bj, on="migration_year", how="left").merge(sh, on="migration_year", how="left")

    if "gdp_before_move" in df.columns:
        df["log_gdp_before"] = np.log(pd.to_numeric(df["gdp_before_move"], errors="coerce").replace({0: np.nan}))
        df["log_gdp_bj"] = np.log(pd.to_numeric(df["gdp_bj"], errors="coerce").replace({0: np.nan}))
        df["log_gdp_sh"] = np.log(pd.to_numeric(df["gdp_sh"], errors="coerce").replace({0: np.nan}))
        df["delta_log_gdp_bj_vs_origin"] = df["log_gdp_bj"] - df["log_gdp_before"]
        df["delta_log_gdp_sh_vs_origin"] = df["log_gdp_sh"] - df["log_gdp_before"]
except Exception:
    pass # 忽略文件或列缺失

# 2.5) 金额类 log1p 变换
for c in ["income_total_m_win","exp_total_m_win","food_exp_m_win","income_to_home_win","rent_m_win"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
        df[f"log1p_{c}"] = np.log1p(df[c])

# =========================
# 3) 标签编码与特征选择
# =========================

# 3.1) 31分类标签 (0..C-1)
codes_all = sorted(df["pro_code"].dropna().astype(int).unique().tolist())
code2idx = {c:i for i,c in enumerate(codes_all)}
idx2code = {i:c for c,i in code2idx.items()}
y31 = df["pro_code"].astype(int).map(code2idx).values
C_31 = len(codes_all) # 31

# 3.2) 特征选择 (selected_features)
core_feats = [
    "male","age","is_han","rural","high_school","junior_college","bachelor","graduate",
    "employed","employment_group","hours_per_week_filled","marriage","length_marriage","kids_number","Happiness",
    "migration_interval",
    "log1p_income_total_m_win","log1p_exp_total_m_win","log1p_food_exp_m_win","log1p_income_to_home_win","log1p_rent_m_win",
    "gdp per capita(k)_b","unemployment(%)_b","population(10k)_b","manageable_income_per_capita_b",
    "education_budget(10k)_b","Medical technicians per 10k_b","road_length_per_10K (km)_b",
    "delta_log_gdp_bj_vs_origin","delta_log_gdp_sh_vs_origin",
    "dist_to_bj","dist_to_sh","dist_bj_minus_sh","dist_min_bj_sh","dist_ratio_bj_over_sh",
]
delta_cols = [c for c in df.columns if c.startswith("delta_")]
feature_pool = set(core_feats) | set(delta_cols)
exclude_cols = {"pro_code","hs_residence","gdp_bj","gdp_sh","log_gdp_before","log_gdp_bj","log_gdp_sh","y_bjsh"}
selected_features = [c for c in feature_pool if c in df.columns and c not in exclude_cols]

# 3.3) 统一数据类型，并识别最终的数值/类别列
X_full = df[selected_features].copy()
obj_cols = X_full.select_dtypes(include=["object"]).columns.tolist()
coerce_numeric_like(X_full, obj_cols) # 尝试转数值

# 最终的类别列 (string/object/category)
cat_cols = []
for c in selected_features:
    if not is_numeric_dtype(X_full[c]):
        cat_cols.append(c)
        X_full[c] = X_full[c].astype("string") # 统一为 string
# 确保 employment_group 是类别
if "employment_group" in selected_features and "employment_group" not in cat_cols:
    cat_cols.append("employment_group")
    X_full["employment_group"] = X_full["employment_group"].astype("string")
    
num_cols = [c for c in selected_features if c not in cat_cols]

print(f"[Info] 最终数值特征数: {len(num_cols)}, 最终类别特征数: {len(cat_cols)}")

# =========================
# 4) 随机划分 (分层，31类为准)
# =========================
X_tr_all, X_te_hold, y_tr_all_31, y_te_31, idx_tr_all, idx_te_hold = train_test_split(
    X_full, y31, np.arange(len(X_full)),
    test_size=0.15, random_state=RANDOM_STATE, stratify=y31
)
X_tr_31, X_va_31, y_tr_31, y_va_31 = train_test_split(
    X_tr_all, y_tr_all_31, 
    test_size=0.1765, random_state=RANDOM_STATE, stratify=y_tr_all_31 # 0.1765 * 0.85 approx 0.15
)

# =========================
# 5) 预处理器构建
# =========================

# 5.1) Scikit-learn Pipeline 预处理器 (用于 Logit, RF, LGBM)
# 类别列需要 OHE
transformers_skl = []
if len(num_cols) > 0:
    transformers_skl.append(("num", SimpleImputer(strategy="median"), num_cols))
if len(cat_cols) > 0:
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
    transformers_skl.append((
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", ohe)
        ]),
        cat_cols
    ))
preprocess_skl = ColumnTransformer(transformers=transformers_skl, remainder="drop")

# 5.2) CatBoost 预处理器 (只需填充缺失值，不OHE)
transformers_cb = []
if len(num_cols) > 0:
    transformers_cb.append(("num", SimpleImputer(strategy="median"), num_cols))
if len(cat_cols) > 0:
    transformers_cb.append(("cat", SimpleImputer(strategy="most_frequent"), cat_cols))
preprocess_cb = ColumnTransformer(transformers=transformers_cb, remainder="drop")

# 5.3) CatBoost 特征索引
cat_feature_indices = list(range(len(num_cols), len(num_cols) + len(cat_cols)))

# 5.4) 计算类别权重 (用于所有模型)
classes = np.unique(y_tr_31)
weights = compute_class_weight(
    class_weight='balanced', 
    classes=classes, 
    y=y_tr_31
)
class_weights_dict = dict(zip(classes, weights))


# =========================
# 6) 模型训练与评估
# =========================

# --- 6.1) Logistic Regression ---
print("\n" + "="*40)
print("=== 6.1) Logistic Regression (31-class) ===")
print("="*40)
logit_pipe = Pipeline([
    ("prep", preprocess_skl),
    ("clf", LogisticRegression(
        multi_class="multinomial", solver="lbfgs", max_iter=1000, 
        class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE
    ))
]).fit(X_tr_31, y_tr_31)

report_metrics_31class("VALID", logit_pipe.predict_proba(X_va_31), y_va_31)
report_metrics_31class("TEST",  logit_pipe.predict_proba(X_te_31), y_te_31)


# --- 6.2) Random Forest ---
print("\n" + "="*40)
print("=== 6.2) Random Forest (31-class) ===")
print("="*40)
rf_pipe = Pipeline([
    ("prep", preprocess_skl),
    ("clf", RandomForestClassifier(
        n_estimators=500, max_depth=None, min_samples_split=5, min_samples_leaf=3,
        max_features="sqrt", class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE
    ))
]).fit(X_tr_31, y_tr_31)

report_metrics_31class("VALID", rf_pipe.predict_proba(X_va_31), y_va_31)
report_metrics_31class("TEST",  rf_pipe.predict_proba(X_te_31), y_te_31)


# --- 6.3) LightGBM ---
print("\n" + "="*40)
print("=== 6.3) LightGBM (31-class) ===")
print("="*40)
lgbm_pipe = Pipeline([
    ("prep", preprocess_skl),
    ("clf", lgb.LGBMClassifier(
        n_estimators=1000, learning_rate=0.05, num_leaves=31, subsample=0.8, colsample_bytree=0.8,
        objective="multiclass", 
        num_class=C_31, # **修复: 明确指定类别数**
        class_weight="balanced", # 使用 balanced 字符串
        n_jobs=-1, random_state=RANDOM_STATE
    ))
]).fit(X_tr_31, y_tr_31)

report_metrics_31class("VALID", lgbm_pipe.predict_proba(X_va_31), y_va_31)
report_metrics_31class("TEST",  lgbm_pipe.predict_proba(X_te_31), y_te_31)


# --- 6.4) CatBoost ---
print("\n" + "="*40)
print("=== 6.4) CatBoost (31-class) ===")
print("="*40)

# CatBoost 训练流程 (手动预处理，以传递 cat_features)
X_tr_cb = preprocess_cb.fit_transform(X_tr_31)
X_va_cb = preprocess_cb.transform(X_va_31)
X_te_cb = preprocess_cb.transform(X_te_31)

catboost_clf = CatBoostClassifier(
    iterations=1000, learning_rate=0.05, depth=6, l2_leaf_reg=3,
    loss_function='MultiClass', 
    eval_metric='Accuracy', # **修复: 使用内置指标**
    class_weights=class_weights_dict, # **修复: 传入数值权重字典**
    cat_features=cat_feature_indices, # 传入类别特征索引
    random_seed=RANDOM_STATE,
    verbose=100, early_stopping_rounds=100, thread_count=-1
)

print(f"[CB Info] 类别特征索引: {cat_feature_indices}")
catboost_clf.fit(
    X_tr_cb, y_tr_31,
    eval_set=(X_va_cb, y_va_31)
)

report_metrics_31class("VALID", catboost_clf.predict_proba(X_va_cb), y_va_31)
report_metrics_31class("TEST",  catboost_clf.predict_proba(X_te_cb), y_te_31)


# --- 最终结果摘要 ---
print("\n" + "="*40)
print("=== 最终测试集性能摘要 ===")
print("="*40)
models = {
    "Logit": logit_pipe.predict_proba(X_te_31),
    "RF": rf_pipe.predict_proba(X_te_31),
    "LGBM": lgbm_pipe.predict_proba(X_te_31),
    "CatBoost": catboost_clf.predict_proba(X_te_cb)
}

for name, proba in models.items():
    res_topk = topk_accuracy(y_te_31, proba, ks=(1, 5))
    y_pred = proba.argmax(axis=1)
    f1_macro = f1_score(y_te_31, y_pred, average='macro', zero_division=0)
    print(f"[{name}] Top-1={res_topk[1]:.4f}, Top-5={res_topk[5]:.4f}, F1-Macro={f1_macro:.4f}")

# 额外：LightGBM 测试集混淆矩阵形状
pred_top1_lgbm = lgbm_pipe.predict_proba(X_te_31).argmax(axis=1)
cm_lgbm = confusion_matrix(y_te_31, pred_top1_lgbm)
print(f"\n[Test] LightGBM Confusion matrix shape: {cm_lgbm.shape} (31x31)")

[Info] 最终数值特征数: 58, 最终类别特征数: 1

=== 6.1) Logistic Regression (31-class) ===


c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[VALID]
  Top-1=0.0848  Top-3=0.2508  Top-5=0.3432  Top-10=0.5456
  F1-Macro=0.0379
[TEST]
  Top-1=0.0853  Top-3=0.2525  Top-5=0.3500  Top-10=0.5531
  F1-Macro=0.0393

=== 6.2) Random Forest (31-class) ===
[VALID]
  Top-1=0.3436  Top-3=0.6049  Top-5=0.7307  Top-10=0.8719
  F1-Macro=0.2525
[TEST]
  Top-1=0.3420  Top-3=0.5994  Top-5=0.7273  Top-10=0.8754
  F1-Macro=0.2543

=== 6.3) LightGBM (31-class) ===
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003499 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2266
[LightGBM] [Info] Number of data points in the train set: 34783, number of used features: 61
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Start training from score -3.401197
[LightGBM] [Info] Start training f

c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[VALID]
  Top-1=0.3900  Top-3=0.6695  Top-5=0.7862  Top-10=0.8987
  F1-Macro=0.2801


...

c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[Logit] Top-1=0.0853, Top-5=0.3500, F1-Macro=0.0393
[RF] Top-1=0.3420, Top-5=0.7273, F1-Macro=0.2543
[LGBM] Top-1=0.3921, Top-5=0.7824, F1-Macro=0.2792
[CatBoost] Top-1=0.3298, Top-5=0.7185, F1-Macro=0.2525


c:\Users\yunzh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



[Test] LightGBM Confusion matrix shape: (30, 30) (31x31)


: 